In [1]:
from pathlib import Path

base_dir = Path.cwd().parent / "데이터수집" / "data"

print("기준 폴더:", base_dir)

if not base_dir.exists():
    raise FileNotFoundError(f"{base_dir} 경로에 data 폴더가 없습니다. 경로를 확인해주세요.")

# demo_output, collection_log.txt 같은 건 제외하고 회차 폴더만 대상으로
session_dirs = sorted([
    d for d in base_dir.iterdir()
    if d.is_dir() and d.name not in ("demo_output", "collection_log.txt")
])

total_meetings = 0

for session_dir in session_dirs:
    # 해당 회차 폴더 안의 모든 xlsx 파일
    xlsx_files = list(session_dir.glob("*.xlsx"))

    # 회의록 ID(prefix) 기준으로 unique 개수 세기
    meeting_ids = set()
    for path in xlsx_files:
        fname = path.name
        if fname.endswith("_minutes_header_summary.xlsx"):
            meeting_ids.add(fname.replace("_minutes_header_summary.xlsx", ""))
        elif fname.endswith("_minutes_speeches.xlsx"):
            meeting_ids.add(fname.replace("_minutes_speeches.xlsx", ""))

    count = len(meeting_ids)
    total_meetings += count

    print(f"- {session_dir.name}: {count}개 회의록 (xlsx 파일 {len(xlsx_files)}개)")

print("\n총 회의록 수:", total_meetings)

기준 폴더: C:\Users\user\Documents\국회회의록API\현웅크롤링\data_collection\data_collection\analysis_scripts\data
- _cache: 0개 회의록 (xlsx 파일 0개)
- _outputs_party_interest: 0개 회의록 (xlsx 파일 0개)
- _outputs_trend: 0개 회의록 (xlsx 파일 0개)
- _outputs_trend_v2: 0개 회의록 (xlsx 파일 0개)
- 제353회: 2개 회의록 (xlsx 파일 4개)
- 제354회: 6개 회의록 (xlsx 파일 12개)
- 제355회: 4개 회의록 (xlsx 파일 8개)
- 제356회: 3개 회의록 (xlsx 파일 6개)
- 제358회: 1개 회의록 (xlsx 파일 2개)
- 제360회: 1개 회의록 (xlsx 파일 2개)
- 제362회: 4개 회의록 (xlsx 파일 8개)
- 제363회: 3개 회의록 (xlsx 파일 6개)
- 제364회: 9개 회의록 (xlsx 파일 18개)
- 제365회: 2개 회의록 (xlsx 파일 4개)
- 제367회: 8개 회의록 (xlsx 파일 16개)
- 제368회: 2개 회의록 (xlsx 파일 4개)
- 제369회: 4개 회의록 (xlsx 파일 8개)
- 제370회: 2개 회의록 (xlsx 파일 4개)
- 제371회: 11개 회의록 (xlsx 파일 22개)
- 제376회: 5개 회의록 (xlsx 파일 10개)
- 제377회: 4개 회의록 (xlsx 파일 8개)
- 제379회: 1개 회의록 (xlsx 파일 2개)
- 제380회: 3개 회의록 (xlsx 파일 6개)
- 제381회: 2개 회의록 (xlsx 파일 4개)
- 제382회: 18개 회의록 (xlsx 파일 36개)
- 제383회: 3개 회의록 (xlsx 파일 6개)
- 제384회: 4개 회의록 (xlsx 파일 8개)
- 제385회: 3개 회의록 (xlsx 파일 6개)
- 제386회: 2개 회의록 (xlsx 파일 4개)
- 제387회: 0개

In [1]:
from pathlib import Path
from dotenv import load_dotenv
import os

# --------------------------------------------------
# 프로젝트 루트(.env) 로드
# --------------------------------------------------
try:
    # .py 실행
    project_root = Path(__file__).resolve().parent.parent
except NameError:
    # Jupyter 실행
    project_root = Path.cwd().parent

env_path = project_root / ".env"
load_dotenv(env_path)

# --------------------------------------------------
# 환경변수 읽기
# --------------------------------------------------
SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

if SUPABASE_URL:
    SUPABASE_URL = SUPABASE_URL.rstrip("/")

# --------------------------------------------------
# 로드 확인
# --------------------------------------------------
if not SUPABASE_URL or not SUPABASE_KEY:
    raise ValueError("❌ .env 파일에서 SUPABASE 환경변수를 불러오지 못했습니다.")

print("✅ SUPABASE_URL:", SUPABASE_URL)
print("✅ SUPABASE_KEY 로드 완료:", SUPABASE_KEY[:10], "...")

✅ SUPABASE_URL: https://oalxysxclyjughtmrvtn.supabase.co
✅ SUPABASE_KEY 로드 완료: sb_publish ...


In [2]:
from pathlib import Path
import time, json, re
from typing import List, Dict, Any, Optional

import numpy as np
import pandas as pd
import requests

# SUPABASE_URL = "https://oalxysxclyjughtmrvtn.supabase.co".rstrip("/")
# SUPABASE_KEY = "sb_publishable_XXXXXXXXXXXXXXXXXXXXXXXX"  # 본인 키

TABLE_NAME = "speeches"
REST_ENDPOINT = f"{SUPABASE_URL}/rest/v1/{TABLE_NAME}"

HEADERS = {
    "apikey": SUPABASE_KEY,
    "Content-Type": "application/json",
    "Prefer": "resolution=merge-duplicates,return=minimal",
}

DISABLE_SSL_VERIFY = False
BATCH_SIZE = 300
MAX_RETRIES = 3
BASE_SLEEP_SEC = 1.0

EXCLUDE_PREFIXES = ("_cache", "_outputs_", "demo_output")
SPEECH_SUFFIX = "_minutes_speeches.xlsx"

ALLOWED_COLS = {
    "speech_id", "meeting_key",
    "session_dir", "meeting_id", "source_file",
    "session", "session_type", "meeting_no", "date",
    "speech_order",
    "speaker_name", "speaker_position", "speaker_area",
    "party", "data_mem_id",
    "agenda_item_orders", "agenda_item_titles", "agenda_item_times",
    "speech_text",
}

# =========================
# Utils
# =========================
def to_text(x: Any) -> Optional[str]:
    if x is None:
        return None
    if isinstance(x, float) and np.isnan(x):
        return None
    try:
        if pd.isna(x):
            return None
    except Exception:
        pass
    if isinstance(x, (list, tuple, dict, set)):
        return json.dumps(x, ensure_ascii=False)
    s = str(x).strip()
    if s == "" or s.lower() in ("nan", "none", "null"):
        return None
    return s

def to_int_safe(x: Any) -> Optional[int]:
    if x is None:
        return None
    if isinstance(x, float) and np.isnan(x):
        return None
    try:
        if pd.isna(x):
            return None
    except Exception:
        pass
    try:
        return int(float(x))
    except Exception:
        return None

def parse_date_any(x):
    if x is None:
        return None
    if isinstance(x, float) and np.isnan(x):
        return None
    try:
        if pd.isna(x):
            return None
    except Exception:
        pass

    if isinstance(x, pd.Timestamp):
        return x.strftime("%Y-%m-%d")

    # 엑셀 시리얼
    if isinstance(x, (int, float)) and not isinstance(x, bool):
        try:
            dt = pd.to_datetime(float(x), unit="D", origin="1899-12-30", errors="coerce")
            if not pd.isna(dt):
                return dt.strftime("%Y-%m-%d")
        except Exception:
            pass

    s = str(x).strip()
    if s == "" or s.lower() in ("nan", "none", "null"):
        return None

    # (목) 등 괄호 제거
    s = re.sub(r"\([^)]*\)", "", s).strip()
    # 요일 접미 제거
    s = re.sub(r"(월요일|화요일|수요일|목요일|금요일|토요일|일요일)\s*$", "", s).strip()
    s = re.sub(r"(월|화|수|목|금|토|일)\s*$", "", s).strip()

    # 한국식/구분자 정리
    s = s.replace("년", "-").replace("월", "-").replace("일", "").strip()
    s = s.replace(".", "-").replace("/", "-")
    s = re.sub(r"\s+", "", s)

    # YYYYMMDD
    if re.fullmatch(r"\d{8}", s):
        s = f"{s[:4]}-{s[4:6]}-{s[6:8]}"

    for fmt in ("%Y-%m-%d", "%Y-%m-%d%H:%M:%S", "%Y-%m-%d%H:%M"):
        dt = pd.to_datetime(s, format=fmt, errors="coerce")
        if not pd.isna(dt):
            return dt.strftime("%Y-%m-%d")

    dt = pd.to_datetime(s, errors="coerce")
    if pd.isna(dt):
        return None
    return dt.strftime("%Y-%m-%d")

def filter_allowed(rec: Dict[str, Any]) -> Dict[str, Any]:
    return {k: v for k, v in rec.items() if k in ALLOWED_COLS}

def chunks(lst: List[Any], n: int):
    for i in range(0, len(lst), n):
        yield lst[i:i + n]

def dedup_records_by_key(records: List[Dict[str, Any]], key: str = "speech_id"):
    m = {}
    for r in records:
        k = r.get(key)
        if k:
            m[k] = r
    return list(m.values())

def postgrest_upsert(records: List[Dict[str, Any]], on_conflict: str = "speech_id") -> None:
    if not records:
        return

    payload = json.dumps(records, ensure_ascii=False)
    last_err = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            r = requests.post(
                REST_ENDPOINT,
                headers=HEADERS,
                params={"on_conflict": on_conflict},
                data=payload,
                timeout=120,
                verify=not DISABLE_SSL_VERIFY,
            )
            if r.status_code in (200, 201, 204):
                return
            last_err = f"[UPSERT FAIL] {r.status_code} {r.text[:2000]}"
        except Exception as e:
            last_err = repr(e)

        time.sleep(BASE_SLEEP_SEC * attempt)

    raise RuntimeError(last_err or "Unknown upsert error")

def build_records_from_speech_xlsx(path: Path, session_dir_name: str) -> List[Dict[str, Any]]:
    df = pd.read_excel(path, engine="openpyxl")
    df.columns = [str(c).strip() for c in df.columns]

    meeting_id = path.name.replace(SPEECH_SUFFIX, "")
    meeting_key = f"{session_dir_name}|{meeting_id}"

    keep_cols = [
        "session", "session_type", "meeting_no", "date",
        "speech_order",
        "speaker_name", "speaker_position", "speaker_area",
        "party", "data_mem_id",
        "agenda_item_orders", "agenda_item_titles", "agenda_item_times",
        "speech_text",
    ]
    present = [c for c in keep_cols if c in df.columns]
    df = df[present].copy()

    if "session" in df.columns:
        df["session"] = pd.to_numeric(df["session"], errors="coerce")

    df["speech_order"] = df["speech_order"].apply(to_int_safe) if "speech_order" in df.columns else None
    df["date"] = df["date"].apply(parse_date_any) if "date" in df.columns else None

    for c in [
        "session_type", "meeting_no",
        "speaker_name", "speaker_position", "speaker_area",
        "party", "data_mem_id",
        "agenda_item_orders", "agenda_item_titles", "agenda_item_times",
        "speech_text",
    ]:
        if c in df.columns:
            df[c] = df[c].apply(to_text)

    df["session_dir"] = session_dir_name
    df["meeting_id"] = meeting_id
    df["meeting_key"] = meeting_key
    df["source_file"] = str(path)

    # speech_id = meeting_key|speech_order (없으면 row index로 보정)
    df = df.reset_index(drop=True)
    df["row_no"] = df.index.astype(int)
    df["speech_id"] = df.apply(
        lambda r: f"{meeting_key}|{r['speech_order']}" if r.get("speech_order") is not None else f"{meeting_key}|r{r['row_no']}",
        axis=1,
    )
    dup_mask = df["speech_id"].duplicated(keep=False)
    if dup_mask.any():
        df.loc[dup_mask, "speech_id"] = df.loc[dup_mask, "row_no"].apply(lambda n: f"{meeting_key}|r{n}")

    df = df.drop(columns=["row_no"], errors="ignore")
    df = df.replace({pd.NA: None})
    df = df.where(pd.notnull(df), None)

    records = [filter_allowed(r) for r in df.to_dict(orient="records")]
    return [r for r in records if r.get("speech_id") and r.get("meeting_key")]

# =========================
# 전체 실행
# =========================
base_dir = Path.cwd() / "data"
print("기준 폴더:", base_dir)

session_dirs = sorted([
    d for d in base_dir.iterdir()
    if d.is_dir() and not d.name.startswith(EXCLUDE_PREFIXES)
])
print("회차 폴더 수:", len(session_dirs))

speech_files: List[Path] = []
for sdir in session_dirs:
    for p in sorted(sdir.glob(f"*{SPEECH_SUFFIX}")):
        if p.name.startswith("~$"):  # ✅ 임시파일 제외
            continue
        speech_files.append(p)

print("발견한 speeches 파일 수:", len(speech_files))

# 연결 점검
test = requests.get(
    f"{SUPABASE_URL}/rest/v1/{TABLE_NAME}?select=speech_id&limit=1",
    headers=HEADERS,
    timeout=30,
    verify=not DISABLE_SSL_VERIFY,
)
print("GET 테스트:", test.status_code, test.text[:200])

loaded_rows = 0
uploaded_rows = 0
failed_files: List[str] = []

t0 = time.time()

for i, fpath in enumerate(speech_files, 1):
    session_dir_name = fpath.parent.name
    try:
        recs = build_records_from_speech_xlsx(fpath, session_dir_name)
        loaded_rows += len(recs)

        for chunk in chunks(recs, BATCH_SIZE):
            chunk = dedup_records_by_key(chunk, "speech_id")
            postgrest_upsert(chunk, on_conflict="speech_id")
            uploaded_rows += len(chunk)

    except Exception as e:
        failed_files.append(f"{str(fpath)} :: {str(e)[:2000]}")

    if i % 10 == 0 or i == len(speech_files):
        elapsed = time.time() - t0
        print(
            f"[{i}/{len(speech_files)}] "
            f"누적 로드: {loaded_rows:,}행 | 누적 업로드: {uploaded_rows:,}행 | "
            f"실패파일: {len(failed_files)} | 경과 {elapsed:,.1f}s"
        )

print("\n=== 완료 ===")
print("총 로드 행:", f"{loaded_rows:,}")
print("총 업로드 행(업서트):", f"{uploaded_rows:,}")
print("실패 파일 수:", len(failed_files))

if failed_files:
    print("\n--- 실패 파일 목록(상위 10개) ---")
    for x in failed_files[:10]:
        print(x)

기준 폴더: C:\Users\user\Documents\국회회의록API\현웅크롤링\data_collection\data_collection\analysis_scripts\data
회차 폴더 수: 66
발견한 speeches 파일 수: 243
GET 테스트: 200 [{"speech_id":"제362회|제3호|793"}]
[10/243] 누적 로드: 3,981행 | 누적 업로드: 3,981행 | 실패파일: 0 | 경과 12.3s
[20/243] 누적 로드: 9,142행 | 누적 업로드: 9,142행 | 실패파일: 0 | 경과 25.3s
[30/243] 누적 로드: 11,460행 | 누적 업로드: 11,460행 | 실패파일: 0 | 경과 32.6s
[40/243] 누적 로드: 15,349행 | 누적 업로드: 15,349행 | 실패파일: 0 | 경과 43.1s
[50/243] 누적 로드: 21,330행 | 누적 업로드: 21,330행 | 실패파일: 0 | 경과 57.4s
[60/243] 누적 로드: 23,611행 | 누적 업로드: 23,611행 | 실패파일: 0 | 경과 64.0s
[70/243] 누적 로드: 25,499행 | 누적 업로드: 25,499행 | 실패파일: 0 | 경과 70.4s
[80/243] 누적 로드: 31,321행 | 누적 업로드: 31,321행 | 실패파일: 0 | 경과 85.7s
[90/243] 누적 로드: 33,022행 | 누적 업로드: 33,022행 | 실패파일: 0 | 경과 91.6s
[100/243] 누적 로드: 35,313행 | 누적 업로드: 35,313행 | 실패파일: 0 | 경과 100.8s
[110/243] 누적 로드: 38,494행 | 누적 업로드: 38,494행 | 실패파일: 0 | 경과 110.6s
[120/243] 누적 로드: 40,576행 | 누적 업로드: 40,576행 | 실패파일: 0 | 경과 120.4s
[130/243] 누적 로드: 42,752행 | 누적 업로드: 42,752행 | 실패파일: 0 | 경과 130.

In [3]:
import requests

q = f"{SUPABASE_URL}/rest/v1/speeches?select=count:count()&date=is.null"
r = requests.get(q, headers=HEADERS, timeout=30)
print(r.status_code, r.text)

400 {"code":"PGRST123","details":null,"hint":null,"message":"Use of aggregate functions is not allowed"}


In [4]:
import requests

q = f"{SUPABASE_URL}/rest/v1/speeches?select=speech_id,meeting_key,date&date=is.null&limit=10"
r = requests.get(q, headers=HEADERS, timeout=30)
print(r.status_code, r.text)

200 [{"speech_id":"__test__|1771504146","meeting_key":"__test_meeting__","date":null}]
